In [1]:
import pandas as pd
from pathlib import Path

base = Path.cwd()
root = base if (base / "data").exists() else base.parent

matches = pd.read_csv(root / "data" / "processed" / "matches_clean.csv", parse_dates=["Date"])
matches = matches.sort_values("Date").reset_index(drop=True)
matches["MatchID"] = matches.index

print(matches.shape)

(1900, 24)


In [2]:
home = pd.DataFrame({
    "MatchID": matches["MatchID"], "Date": matches["Date"], "IsHome": 1,
    "Team": matches["HomeTeam"],
    "GF": matches["FTHG"], "GA": matches["FTAG"],
    "Points": matches["Target"].map({0: 3, 1: 1, 2: 0}),
    "SOT": matches["HST"],
})
away = pd.DataFrame({
    "MatchID": matches["MatchID"], "Date": matches["Date"], "IsHome": 0,
    "Team": matches["AwayTeam"],
    "GF": matches["FTAG"], "GA": matches["FTHG"],
    "Points": matches["Target"].map({2: 3, 1: 1, 0: 0}),
    "SOT": matches["AST"],
})

long = pd.concat([home, away]).sort_values(["Team", "Date", "MatchID"]).reset_index(drop=True)
print(long.shape)
long.head()

(3800, 8)


,MatchID,Date,IsHome,Team,GF,GA,Points,SOT
0,0,2021-08-13,0,Arsenal,0,2,0,4
1,18,2021-08-22,1,Arsenal,0,2,0,3
2,22,2021-08-28,0,Arsenal,0,5,0,0
3,31,2021-09-11,1,Arsenal,1,0,3,7
4,46,2021-09-18,0,Arsenal,1,0,3,3


In [3]:
feat_cols = ["GF", "GA", "Points", "SOT"]
g = long.groupby("Team")

for c in feat_cols:
    # shift(1) = use only matches BEFORE this one (no data leakage)
    long[f"{c}_form5"] = g[c].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())

long[long["Team"] == "Arsenal"][["Date", "Team", "GF", "Points", "GF_form5", "Points_form5"]].head(10)

,Date,Team,GF,Points,GF_form5,Points_form5
0,2021-08-13,Arsenal,0,0,NaN,NaN
1,2021-08-22,Arsenal,0,0,NaN,NaN
2,2021-08-28,Arsenal,0,0,NaN,NaN
3,2021-09-11,Arsenal,1,3,0.00,0.00
4,2021-09-18,Arsenal,1,3,0.25,0.75
5,2021-09-26,Arsenal,3,3,0.40,1.20
6,2021-10-02,Arsenal,0,1,1.00,1.80
7,2021-10-18,Arsenal,2,1,1.00,2.00
8,2021-10-22,Arsenal,3,3,1.40,2.20
9,2021-10-30,Arsenal,2,3,1.80,2.20


In [4]:
form_cols = [f"{c}_form5" for c in feat_cols]

home_f = long[long["IsHome"] == 1].set_index("MatchID")[form_cols].add_prefix("Home_")
away_f = long[long["IsHome"] == 0].set_index("MatchID")[form_cols].add_prefix("Away_")

data = matches.set_index("MatchID").join(home_f).join(away_f).reset_index()

# Home minus away: positive = home team in better form
data["Points_form_diff"] = data["Home_Points_form5"] - data["Away_Points_form5"]
data["GF_form_diff"] = data["Home_GF_form5"] - data["Away_GF_form5"]
data["GA_form_diff"] = data["Home_GA_form5"] - data["Away_GA_form5"]
data["SOT_form_diff"] = data["Home_SOT_form5"] - data["Away_SOT_form5"]

feature_cols = [c for c in data.columns if c.startswith(("Home_", "Away_")) or c.endswith("_form_diff")]

model_df = data.dropna(subset=feature_cols).reset_index(drop=True)
print("Rows before:", len(data), "| after dropping early-season gaps:", len(model_df))
print("Features:", len(feature_cols))

Rows before: 1900 | after dropping early-season gaps: 1849
Features: 12


In [5]:
keep = ["Date", "Season", "HomeTeam", "AwayTeam", "Result", "Target"] + feature_cols
model_df[keep].to_csv(root / "data" / "processed" / "model_features.csv", index=False)
print("Saved", model_df[keep].shape)

Saved (1849, 18)
